In [1]:
import torch
from torch.utils.data import DataLoader, random_split
import numpy as np

import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
import sys
from pathlib import Path
GAN_CLASSES = [1, 3]
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.data.H5_dataset import EEGH5Dataset
from src.data.preprocessing import load_multiple_subjects
from src.data.class_dataset import EEGClassDataset
from src.utils.logging import get_logger
from src.utils.config import (
    LATENT_DIM,
    WINDOW_SIZE,
    WINDOW_STRIDE,
    BATCH_SIZE,
    WANTED_CHANNELS,
    NUM_CHANNELS,
    LR_CLASSIFIER,
    EPOCHS_CLASSIFIER
)
from src.models.classifier import Classifier
from src.training.train_classifier import train_classifier

In [2]:
logger = get_logger("TRTR")

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {device}")

2026-08-06 10:46:01 | INFO | Using device: cuda


In [4]:

dataset = EEGH5Dataset("C:\\Users\\danie\\Documents\\Tesis\\Prueba\\BestTimeGAN\\data\\processed\\eeg_train_8.h5", keep_classes=GAN_CLASSES)
val_dataset = EEGH5Dataset("C:\\Users\\danie\\Documents\\Tesis\\Prueba\\BestTimeGAN\\data\\processed\\eeg_val_8.h5", keep_classes=GAN_CLASSES)
test_dataset= EEGH5Dataset("C:\\Users\\danie\\Documents\\Tesis\\Prueba\\BestTimeGAN\\data\\processed\\eeg_test_8.h5", keep_classes=GAN_CLASSES)

In [5]:
train_dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
)

logger.info(f"Dataset windows: {len(dataset)}")

2026-08-06 10:46:01 | INFO | Dataset windows: 7992


In [6]:
val_dataloader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
)

logger.info(f"Dataset windows: {len(val_dataset)}")

2026-08-06 10:46:01 | INFO | Dataset windows: 888


In [7]:
test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
)

logger.info(f"Dataset windows: {len(test_dataset)}")

2026-08-06 10:46:01 | INFO | Dataset windows: 888


In [8]:
C = Classifier(9,512,2)

In [9]:
orig_labels_map = torch.tensor(GAN_CLASSES, dtype=torch.long).to(device) if GAN_CLASSES is not None else None

In [10]:
train_classifier(
    C,
    train_dataloader,
    val_dataloader,
    EPOCHS_CLASSIFIER,
    LR_CLASSIFIER,
    device,
    orig_labels_map,
    logger
)

2026-08-06 10:46:03 | INFO | Epoch 1 | Train Loss: 0.6986 | Train Acc: 0.5152 | Val Loss: 0.6879 | Val Acc: 0.5417
2026-08-06 10:46:04 | INFO | Epoch 2 | Train Loss: 0.6742 | Train Acc: 0.5763 | Val Loss: 0.6649 | Val Acc: 0.6088
2026-08-06 10:46:06 | INFO | Epoch 3 | Train Loss: 0.6310 | Train Acc: 0.6437 | Val Loss: 0.6520 | Val Acc: 0.6215
2026-08-06 10:46:07 | INFO | Epoch 4 | Train Loss: 0.5793 | Train Acc: 0.6950 | Val Loss: 0.6067 | Val Acc: 0.6968
2026-08-06 10:46:08 | INFO | Epoch 5 | Train Loss: 0.5381 | Train Acc: 0.7359 | Val Loss: 0.6136 | Val Acc: 0.6690
2026-08-06 10:46:09 | INFO | Epoch 6 | Train Loss: 0.5191 | Train Acc: 0.7456 | Val Loss: 0.6103 | Val Acc: 0.6586
2026-08-06 10:46:10 | INFO | Epoch 7 | Train Loss: 0.5060 | Train Acc: 0.7521 | Val Loss: 0.5855 | Val Acc: 0.6956
2026-08-06 10:46:11 | INFO | Epoch 8 | Train Loss: 0.4944 | Train Acc: 0.7636 | Val Loss: 0.6074 | Val Acc: 0.6562
2026-08-06 10:46:12 | INFO | Epoch 9 | Train Loss: 0.4883 | Train Acc: 0.7626 | 

Classifier(
  (temporal): Conv2d(1, 8, kernel_size=(1, 64), stride=(1, 1), padding=(0, 32), bias=False)
  (spatial): Conv2d(8, 16, kernel_size=(9, 1), stride=(1, 1), groups=8, bias=False)
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (pool1): AvgPool2d(kernel_size=(1, 8), stride=(1, 8), padding=0)
  (dropout): Dropout(p=0.25, inplace=False)
  (sep): Conv2d(16, 16, kernel_size=(1, 16), stride=(1, 1), padding=(0, 8), groups=16, bias=False)
  (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (pool2): AvgPool2d(kernel_size=(1, 8), stride=(1, 8), padding=0)
  (fc): Linear(in_features=128, out_features=2, bias=True)
)

In [11]:
torch.save(C.state_dict(), "TRClassifier2.pt")

In [12]:
C.load_state_dict(torch.load("TRClassifier2.pt", weights_only=True))

<All keys matched successfully>

In [16]:
if orig_labels_map is not None:
    _inv_map = {int(v): i for i, v in enumerate(orig_labels_map.tolist())}
    def _to_local(raw_labels: torch.Tensor) -> torch.Tensor:
        """Remap raw dataset labels to 0-based local indices for Classifier."""
        return torch.tensor(
            [_inv_map[int(l)] for l in raw_labels.tolist()],
            dtype=torch.long,
            device=raw_labels.device,
        )
else:
    def _to_local(raw_labels):
        return raw_labels

In [17]:
def evaluate(model, dataloader, device):
    model.to(device)
    model.eval()  # switch to inference mode


    total = 0
    correct = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():  # disable gradients
        for x, y in dataloader:
            y = _to_local(y)
            x = x.to(device)
            y = y.to(device)

            logits = model(x)              # (B, num_classes)
            preds = torch.argmax(logits, dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

            all_preds.append(preds.cpu())
            all_labels.append(y.cpu())

    accuracy = correct / total

    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)

    return accuracy, all_preds, all_labels

In [18]:
accuracy, all_preds, all_labels = evaluate(C, test_dataloader, device)

In [19]:
from sklearn.metrics import classification_report

print(classification_report(all_labels, all_preds))


              precision    recall  f1-score   support

           0       0.83      0.45      0.58       434
           1       0.62      0.91      0.74       430

    accuracy                           0.68       864
   macro avg       0.73      0.68      0.66       864
weighted avg       0.73      0.68      0.66       864

